# New Section
VGG16 + SVM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install git+https://github.com/AlpacaDB/selectivesearch


1. Prétraitement des Images

In [ ]:
import concurrent.futures
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from skimage import img_as_float
from skimage.measure import regionprops
from skimage.segmentation import felzenszwalb
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report


# Charger les images et les étiquettes
def load_images_and_labels(directory):
    images = []
    labels = []
    for file in os.listdir(directory):
        if file.endswith('_gtFine_color.png'):
            img_path = os.path.join(directory, file)
            lbl_path = os.path.join(directory, file.replace('_gtFine_color.png', '_gtFine_labelIds.png'))

            img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)  # Charger l'image avec 4 canaux
            lbl = cv2.imread(lbl_path, cv2.IMREAD_UNCHANGED)  # Charger les labels avec 3 canaux

            if img.shape[2] == 4:  # Si l'image a 4 canaux, enlever le canal alpha
                img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

            images.append(img)
            labels.append(lbl)

    return images, labels

# Chemins vers les répertoires des images et des annotations
train_dir = '/content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine/train/aachen'
val_dir = '/content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine/val/lindau'

# Chargement des données d'entraînement et de validation
images_train, labels_train = load_images_and_labels(train_dir)
images_val, labels_val = load_images_and_labels(val_dir)

# Redimensionner et normaliser les images
def preprocess_images(images, target_size=(256, 256)):
    processed_images = []
    for img in images:
        resized_img = cv2.resize(img, target_size)
        normalized_img = resized_img / 255.0
        processed_images.append(normalized_img)
    return np.array(processed_images)

def preprocess_labels(labels, target_size=(256, 256)):
    processed_labels = []
    for lbl in labels:
        resized_lbl = cv2.resize(lbl, target_size, interpolation=cv2.INTER_NEAREST)
        processed_labels.append(resized_lbl)
    return np.array(processed_labels)

preprocessed_images_train = preprocess_images(images_train)
preprocessed_images_val = preprocess_images(images_val)
preprocessed_labels_train = preprocess_labels(labels_train)
preprocessed_labels_val = preprocess_labels(labels_val)

2. Propositions de Régions et Extraction de Caractéristiques

.Utilisation de l'algorithme de recherche sélective pour générer des régions candidates.

.Utilisation d'un réseau de neurones convolutif pré-entraîné (VGG16) pour extraire des caractéristiques.

Sans le traitement par lots

Temps d'exécution trop long

In [ ]:
# Utiliser skimage pour la recherche sélective
def get_region_proposals(image):
    segments = felzenszwalb(img_as_float(image), scale=100, sigma=0.5, min_size=50)
    regions = regionprops(segments)
    candidates = []
    for region in regions:
        minr, minc, maxr, maxc = region.bbox
        w = maxc - minc
        h = maxr - minr
        if w > 0 and h > 0:
            candidates.append((minc, minr, w, h))
    return candidates

regions_proposals_train = [get_region_proposals(img) for img in preprocessed_images_train]
regions_proposals_val = [get_region_proposals(img) for img in preprocessed_images_val]

# Charger le modèle pré-entraîné VGG16
model = VGG16(weights='imagenet', include_top=False, input_shape=(256, 256, 3))

def extract_features(image, regions):
    features = []
    img_h, img_w, _ = image.shape
    for (x, y, w, h) in regions:
        # S'assurer que les régions sont dans les limites de l'image
        x_end = min(x + w, img_w)
        y_end = min(y + h, img_h)

        roi = image[y:y_end, x:x_end]

        # Redimensionner seulement si la région est valide
        if roi.shape[0] > 0 and roi.shape[1] > 0:
            roi_resized = cv2.resize(roi, (256, 256))
            roi_array = np.expand_dims(roi_resized, axis=0)
            roi_preprocessed = preprocess_input(roi_array)
            feature = model.predict(roi_preprocessed)
            features.append(feature)
    return np.array(features)

image_features_train = [extract_features(img, regions) for img, regions in zip(preprocessed_images_train, regions_proposals_train)]
image_features_val = [extract_features(img, regions) for img, regions in zip(preprocessed_images_val, regions_proposals_val)]


Streaming output truncated to the last 5000 lines.
1/1 [==============================] - 1s 704ms/step


KeyboardInterrupt: 

Traitement par lots à utiliser

In [ ]:
# Fonction pour obtenir les propositions de régions
def get_region_proposals(image):
    segments = felzenszwalb(img_as_float(image), scale=200, sigma=0.9, min_size=100)
    regions = regionprops(segments)
    candidates = []
    for region in regions:
        minr, minc, maxr, maxc = region.bbox
        w = maxc - minc
        h = maxr - minr
        if w > 0 and h > 0:
            candidates.append((minc, minr, w, h))
    return candidates

# Charger le modèle pré-entraîné VGG16
model = VGG16(weights='imagenet', include_top=False, input_shape=(256, 256, 3))

@tf.function
def extract_features_batch(images_batch):
    preprocessed_batch = preprocess_input(images_batch)
    return model(preprocessed_batch)

# Fonction pour extraire les caractéristiques d'une image
def extract_features(image, regions, batch_size=8):
    features = []
    img_h, img_w, _ = image.shape
    batch_images = []
    for (x, y, w, h) in regions:
        x_end = min(x + w, img_w)
        y_end = min(y + h, img_h)
        roi = image[y:y_end, x:x_end]
        if roi.shape[0] > 0 and roi.shape[1] > 0:
            roi_resized = cv2.resize(roi, (256, 256))
            batch_images.append(roi_resized)
        if len(batch_images) == batch_size:
            batch_array = np.array(batch_images)
            features_batch = extract_features_batch(batch_array)
            features.extend(features_batch)
            batch_images = []

    if batch_images:
        batch_array = np.array(batch_images)
        features_batch = extract_features_batch(batch_array)
        features.extend(features_batch)

    return np.array(features)

# Traiter les images d'entraînement et de validation par lots
def process_images(images, regions_proposals, batch_size=32):
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [executor.submit(extract_features, img, regions, batch_size) for img, regions in zip(images, regions_proposals)]
        return [future.result() for future in futures]

regions_proposals_train = [get_region_proposals(img) for img in preprocessed_images_train]
regions_proposals_val = [get_region_proposals(img) for img in preprocessed_images_val]

image_features_train = process_images(preprocessed_images_train, regions_proposals_train)
image_features_val = process_images(preprocessed_images_val, regions_proposals_val)

# Extraire les caractéristiques et les labels pour l'entraînement et la validation
def extract_features_and_labels(image_features, regions_proposals, labels):
    X = []
    y = []
    for img_features, lbl, regions in zip(image_features, labels, regions_proposals):
        for feature, (x, y, w, h) in zip(img_features, regions):
            roi_label = lbl[y:y+h, x:x+w]
            pedestrian_label = 1 if np.sum(roi_label == 24) > 0 else 0  # 24: Label pour piéton dans Cityscapes
            X.append(feature.flatten())
            y.append(pedestrian_label)
    return np.array(X), np.array(y)

X_train, y_train = extract_features_and_labels(image_features_train, regions_proposals_train, preprocessed_labels_train)
X_val, y_val = extract_features_and_labels(image_features_val, regions_proposals_val, preprocessed_labels_val)

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)

In [ ]:
# Encodage des labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

# Entraîner le classificateur SVM
clf = SVC(kernel='linear', probability=True)
clf.fit(X_train, y_train_encoded)

# Prédictions et scores pour les régions
y_pred_train = clf.predict(X_train)
y_pred_val = clf.predict(X_val)
y_pred_train_scores = clf.predict_proba(X_train)[:, 1]
y_pred_val_scores = clf.predict_proba(X_val)[:, 1]

print("Classification Report for Training Set")
print(classification_report(y_train_encoded, y_pred_train))

print("Classification Report for Validation Set")
print(classification_report(y_val_encoded, y_pred_val))

In [ ]:

# Post-traitement avec suppression non maximale
def non_maximum_suppression(boxes, scores, threshold=0.3):
    if len(boxes) == 0:
        return []

    boxes = np.array(boxes)
    scores = np.array(scores)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 0] + boxes[:, 2]
    y2 = boxes[:, 1] + boxes[:, 3]

    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0, xx2 - xx1 + 1)
        h = np.maximum(0, yy2 - yy1 + 1)
        inter = w * h

        ovr = inter / (areas[i] + areas[order[1:]] - inter)
        inds = np.where(ovr <= threshold)[0]
        order = order[inds + 1]

    return keep

# Appliquer la suppression non maximale aux résultats
def apply_nms_to_results(results, scores):
    return non_maximum_suppression(results, scores)

nms_results_train = [apply_nms_to_results(result, score) for result, score in zip(regions_proposals_train, y_pred_train_scores)]
nms_results_val = [apply_nms_to_results(result, score) for result, score in zip(regions_proposals_val, y_pred_val_scores)]

In [ ]:
# Visualiser les résultats
def visualize_results(images, labels, proposals, nms_results, title):
    for img, lbl, props, nms_res in zip(images, labels, proposals, nms_results):
        plt.figure(figsize=(15, 6))
        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title('Image')

        plt.subplot(1, 3, 2)
        plt.imshow(lbl)
        plt.title('Label')

        plt.subplot(1, 3, 3)
        plt.imshow(img)
        for i in nms_res:
            (x, y, w, h) = props[i]
            plt.gca().add_patch(plt.Rectangle((x, y), w, h, edgecolor='red', facecolor='none', linewidth=2))
        plt.title('Predictions (NMS)')

        plt.suptitle(title)
        plt.show()

visualize_results(preprocessed_images_train, preprocessed_labels_train, regions_proposals_train, nms_results_train, 'Training Set Results')
visualize_results(preprocessed_images_val, preprocessed_labels_val, regions_proposals_val, nms_results_val, 'Validation Set Results')


# New Section

YOLOv5

In [ ]:
# Installer les dépendances et cloner le dépôt YOLOv5
!pip install torch torchvision opencv-python matplotlib
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

In [ ]:
import os

def check_directory_structure(base_dir):
    # Dictionnaire pour stocker les résultats
    structure_check = {
        "train_images": False,
        "train_labels": False,
        "val_images": False,
        "val_labels": False
    }

    # Chemins attendus
    train_images_dir = os.path.join(base_dir, 'train/images')
    train_labels_dir = os.path.join(base_dir, 'train/labels')
    val_images_dir = os.path.join(base_dir, 'val/images')
    val_labels_dir = os.path.join(base_dir, 'val/labels')

    # Vérifiez la présence des répertoires
    structure_check["train_images"] = os.path.isdir(train_images_dir)
    structure_check["train_labels"] = os.path.isdir(train_labels_dir)
    structure_check["val_images"] = os.path.isdir(val_images_dir)
    structure_check["val_labels"] = os.path.isdir(val_labels_dir)

    # Vérifiez la présence des fichiers d'images et de labels
    if structure_check["train_images"]:
        train_image_files = os.listdir(train_images_dir)
        print(f"Found {len(train_image_files)} training images.")

    if structure_check["train_labels"]:
        train_label_files = os.listdir(train_labels_dir)
        print(f"Found {len(train_label_files)} training labels.")

    if structure_check["val_images"]:
        val_image_files = os.listdir(val_images_dir)
        print(f"Found {len(val_image_files)} validation images.")

    if structure_check["val_labels"]:
        val_label_files = os.listdir(val_labels_dir)
        print(f"Found {len(val_label_files)} validation labels.")

    return structure_check

# Base directory path
base_dir = '/content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine/processed_data'
structure_check = check_directory_structure(base_dir)

print("Directory structure check results:")
print(structure_check)


Found 2975 training images.
Found 2975 training labels.
Found 500 validation images.
Found 500 validation labels.
Directory structure check results:
{'train_images': True, 'train_labels': True, 'val_images': True, 'val_labels': True}


In [ ]:
import os

def list_files(base_dir, split):
    split_dir = os.path.join(base_dir, split)
    files = os.listdir(split_dir)
    print(f"Files in {split} directory:")
    for file in files:
        print(file)

# Chemins vers les répertoires des images et des labels
base_dir = '/content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine'

list_files(base_dir, 'train')
list_files(base_dir, 'val')


Files in train directory:
aachen
bochum
cologne
bremen
darmstadt
dusseldorf
hamburg
erfurt
strasbourg
ulm
tubingen
monchengladbach
krefeld
jena
hanover
stuttgart
weimar
zurich
aachen.cache
Files in val directory:
frankfurt
lindau
munster


In [ ]:
import os
import json
import cv2

# Chemins vers les répertoires des images et des labels
train_dir = '/content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine/train'
val_dir = '/content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine/val'
output_dir = '/content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine/processed_data'

# Créez les répertoires de sortie si nécessaires
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'train/images'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'train/labels'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'val/images'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'val/labels'), exist_ok=True)

def process_images_and_labels(base_dir, split):
    output_image_dir = os.path.join(output_dir, f'{split}/images')
    output_label_dir = os.path.join(output_dir, f'{split}/labels')

    for city in os.listdir(base_dir):
        city_dir = os.path.join(base_dir, city)
        if os.path.isdir(city_dir):
            for filename in os.listdir(city_dir):
                if filename.endswith("_color.png"):
                    image_path = os.path.join(city_dir, filename)
                    label_path = os.path.join(city_dir, filename.replace("_color.png", "_polygons.json"))
                    if not os.path.exists(label_path):
                        continue

                    # Copiez l'image dans le répertoire de sortie
                    output_image_path = os.path.join(output_image_dir, filename)
                    cv2.imwrite(output_image_path, cv2.imread(image_path))

                    # Convertir les annotations en format YOLO
                    with open(label_path) as f:
                        data = json.load(f)

                    height, width, _ = cv2.imread(output_image_path).shape
                    yolo_annotations = []
                    for obj in data['objects']:
                        if obj['label'] == 'pedestrian':
                            polygon = obj['polygon']
                            x_coords = [point[0] for point in polygon]
                            y_coords = [point[1] for point in polygon]
                            x_center = (max(x_coords) + min(x_coords)) / 2 / width
                            y_center = (max(y_coords) + min(y_coords)) / 2 / height
                            bbox_width = (max(x_coords) - min(x_coords)) / width
                            bbox_height = (max(y_coords) - min(y_coords)) / height
                            yolo_annotations.append(f"0 {x_center} {y_center} {bbox_width} {bbox_height}")

                    output_label_path = os.path.join(output_label_dir, filename.replace("_color.png", ".txt"))
                    with open(output_label_path, 'w') as f:
                        f.write("\n".join(yolo_annotations))

# Traitez les données d'entraînement et de validation
process_images_and_labels(train_dir, 'train')
process_images_and_labels(val_dir, 'val')

# Vérifiez la structure des répertoires après la transformation
def check_directory_structure(base_dir):
    structure = {
        'train_images': os.path.exists(os.path.join(base_dir, 'train/images')),
        'train_labels': os.path.exists(os.path.join(base_dir, 'train/labels')),
        'val_images': os.path.exists(os.path.join(base_dir, 'val/images')),
        'val_labels': os.path.exists(os.path.join(base_dir, 'val/labels'))
    }
    print(f"Directory structure check results: {structure}")
    return structure

# Vérification de la structure des répertoires
check_directory_structure(output_dir)


Directory structure check results: {'train_images': True, 'train_labels': True, 'val_images': True, 'val_labels': True}


{'train_images': True,
 'train_labels': True,
 'val_images': True,
 'val_labels': True}

In [ ]:
# Créer le fichier YAML de configuration du dataset
yaml_content = """
train: /content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine/processed_data/train
val: /content/drive/MyDrive/projet/gtFine_trainvaltest/gtFine/processed_data/val

nc: 1  # Nombre de classes (piétons)
names: ['pedestrian']  # Noms des classes
"""

# Écrire le fichier YAML dans Google Colab
with open('data/cityscapes.yaml', 'w') as file:
    file.write(yaml_content)

print("Fichier de configuration YAML créé.")

Fichier de configuration YAML créé.


In [ ]:
# Entraînement du modèle YOLOv5 avec les données préparées
!python train.py --img 640 --batch 16 --epochs 50 --data data/cityscapes.yaml --cfg yolov5s.yaml --weights yolov5s.pt --name yolov5_cityscapes


2024-06-12 15:33:40.642591: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-12 15:33:40.642692: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-12 15:33:40.813485: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
train: weights=yolov5s.pt, cfg=yolov5s.yaml, data=data/cityscapes.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=50, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, 

In [ ]:
# Évaluation du modèle YOLOv5
!python val.py --data data/cityscapes.yaml --weights runs/train/yolov5_cityscapes/weights/best.pt --img 640

# Afficher les résultats
from IPython.display import Image, display
display(Image(filename='runs/train/yolov5_cityscapes/results.png'))
